# Module 02 live coding: tidy data and codebook

Goal: turn a wide learner table into a tidy analysis table, then create a codebook and a first figure.

## 1. Load tools and paths
We keep input files in `data/raw` and save paper-facing outputs into `outputs`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

BASE = Path.cwd()
RAW = BASE / "data" / "raw"
OUT_TABLES = BASE / "outputs" / "tables"
OUT_FIGURES = BASE / "outputs" / "figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)


## 2. Read a wide table
The learner survey is wide because pre/post measures are stored as separate columns.

In [ ]:
wide = pd.read_csv(RAW / "module02_toy_learner_survey_wide.csv")
print(wide.shape)
wide.head()


## 3. Identify variable types before plotting
A codebook row should exist for every important variable.

In [ ]:
codebook = pd.read_csv(OUT_TABLES / "module02_codebook.csv")
codebook[["variable_name", "variable_type", "role", "unit_of_observation"]]


## 4. Reshape wide scores into tidy long format
After reshaping, one row means one learner, one skill, and one time point.

In [ ]:
vocab = wide[["learner_id", "research_track", "activity_group", "cefr_start", "pre_vocab_score", "post_vocab_score"]].melt(
    id_vars=["learner_id", "research_track", "activity_group", "cefr_start"],
    value_vars=["pre_vocab_score", "post_vocab_score"],
    var_name="measure_time",
    value_name="score"
)
vocab["skill"] = "vocabulary"
vocab["time"] = vocab["measure_time"].str.extract(r"^(pre|post)")

speaking = wide[["learner_id", "research_track", "activity_group", "cefr_start", "pre_speaking_score", "post_speaking_score"]].melt(
    id_vars=["learner_id", "research_track", "activity_group", "cefr_start"],
    value_vars=["pre_speaking_score", "post_speaking_score"],
    var_name="measure_time",
    value_name="score"
)
speaking["skill"] = "speaking"
speaking["time"] = speaking["measure_time"].str.extract(r"^(pre|post)")

tidy_scores = pd.concat([vocab, speaking], ignore_index=True)[[
    "learner_id", "research_track", "activity_group", "cefr_start", "skill", "time", "score"
]]
tidy_scores["time"] = pd.Categorical(tidy_scores["time"], categories=["pre", "post"], ordered=True)
tidy_scores.sort_values(["learner_id", "skill", "time"]).head(8)


## 5. Export the tidy table
A paper workflow leaves behind a table that other people can inspect.

In [ ]:
tidy_scores.sort_values(["learner_id", "skill", "time"]).to_csv(
    OUT_TABLES / "module02_tidy_learner_scores_long.csv",
    index=False
)
summary = tidy_scores.groupby(["skill", "time"], observed=True, as_index=False)["score"].mean()
summary


## 6. Draw a first figure from tidy data
Because the table has `skill`, `time`, and `score`, the plot code is direct.

In [ ]:
colors = {"vocabulary": "#245ee8", "speaking": "#16815f"}
fig, ax = plt.subplots(figsize=(8.8, 5.2))
for skill, group in summary.groupby("skill"):
    x = [0 if t == "pre" else 1 for t in group["time"].astype(str)]
    ax.plot(x, group["score"], marker="o", linewidth=2.8, markersize=8, label=skill, color=colors[skill])

ax.set_xticks([0, 1], ["pre", "post"])
ax.set_ylabel("Mean score")
ax.set_xlabel("Measurement time")
ax.set_title("Tidy data makes a pre/post figure straightforward", loc="left", weight="bold")
ax.grid(axis="y", color="#e6edf7")
ax.legend(frameon=False, title="Skill")

for ext in ["png", "svg", "pdf"]:
    fig.savefig(OUT_FIGURES / f"module02_tidy_prepost_example.{ext}", dpi=220, bbox_inches="tight")
plt.show()


## 7. Caption draft
A caption should state the unit of observation after reshaping.

In [ ]:
caption = (
    "Figure 1. Mean learner scores after reshaping wide pre/post columns into a tidy learner-skill-time table. "
    "Each row in the analysis table represents one learner, one skill, and one time point. "
    "The figure is appropriate only after the codebook defines skill, time, and score; "
    "it should be read as a teaching example rather than causal evidence because the dataset is synthetic."
)
(OUT_TABLES / "module02_caption_draft.md").write_text(caption, encoding="utf-8")
print(caption)
